# Riyadh Livability Index (RLI) — ML Engine
---
**Project:** Neighborhood DNA — The 15-Minute City Index  
**Input:** `Riyadh_Master_Dataset.csv` — 348K property rows × 29 columns  

**Three ML Systems:**
1. **Unsupervised Clustering** — K-Means vs Hierarchical (k=2,3,5) with Silhouette, Calinski-Harabasz, Davies-Bouldin metrics
2. **Supervised Regression** — Linear, Ridge, Random Forest, Gradient Boosting for price prediction (R², MAE, RMSE, MAPE)
3. **Supervised Classification** — Logistic Regression, Random Forest, Gradient Boosting for price tier prediction (Accuracy, Precision, Recall, F1, Confusion Matrix)

Plus the **Recommendation Engine** using K-Means cluster matching.

## 1 | Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                              GradientBoostingRegressor, GradientBoostingClassifier)
from sklearn.metrics import (silhouette_score, calinski_harabasz_score, davies_bouldin_score,
                             r2_score, mean_absolute_error, mean_squared_error,
                             accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)
from math import pi

warnings.filterwarnings("ignore")

plt.rcParams.update({
    'figure.facecolor': '#0a0e27', 'axes.facecolor': '#0a0e27',
    'axes.edgecolor': '#2a2f4e', 'axes.labelcolor': '#c4c7d4',
    'text.color': '#c4c7d4', 'xtick.color': '#8b8fa3', 'ytick.color': '#8b8fa3',
    'grid.color': '#1a1f3e', 'grid.alpha': 0.5, 'font.family': 'sans-serif',
    'font.size': 11, 'figure.dpi': 120, 'figure.figsize': (14, 6)
})

GOLD='#f0c05a'; CYAN='#4fc3f7'; CORAL='#ff6b6b'; MINT='#66bb6a'; PURPLE='#ab47bc'
PALETTE = [GOLD, CYAN, CORAL, MINT, PURPLE, '#ff8a65', '#42a5f5', '#ef5350']

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Setup complete.')

In [ ]:
url = 'https://raw.githubusercontent.com/AbdulrahmanB-25/Machine_Learning_Competition/main/Riyadh_Master_Dataset.csv'
try:
    df_raw = pd.read_csv(url)
    print(f"Loaded from GitHub: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
except:
    df_raw = pd.read_csv('Riyadh_Master_Dataset.csv')
    print(f"Loaded locally: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Unique neighborhoods: {df_raw['neighborhood'].nunique()}")
df_raw.head(3)

## 2 | Constants & Configuration

In [ ]:
EPSILON = 1e-9
H_MAX   = np.log(4)          # ln(4) — theoretical max Shannon entropy for 4 pillars
K_BEST  = 5                  # optimal cluster count from silhouette analysis

CATEGORY_MAP = {
    1:  'Apartment (Rent)',   2:  'Land',            3:  'Villa',
    4:  'Floor (Rent)',       5:  'Duplex (Rent)',    6:  'Apartment (Sale)',
    7:  'Commercial Land',    8:  'Office',           9:  'Building',
    10: 'Compound',           11: 'Farm',             13: 'Room',
    14: 'Shop',               15: 'Warehouse',        16: 'Commercial Building',
    17: 'Tower',              18: 'Camp',             19: 'Parking',
    20: 'Studio',             21: 'Chalet',           22: 'Duplex (Sale)',
    23: 'Rest House',         24: 'Palace',
}

# Categories where room filtering makes no sense
NO_ROOM_CATEGORIES = {
    'Land', 'Commercial Land', 'Farm', 'Rest House', 'Parking',
    'Warehouse', 'Camp', 'Shop',
}

# ── 4 RLI Pillars ──
PILLARS = {
    'Core': {
        'weight': 0.40,
        'features': ['med_facilities', 'edu_primary', 'essential_retail', 'religious'],
    },
    'Mobility': {
        'weight': 0.25,
        'features': ['bus_count', 'metro_count', 'pedestrian', 'connectivity_score'],
    },
    'Well-being': {
        'weight': 0.20,
        'features': ['dining_cafe', 'parks_green', 'sports_play', 'fitness_care'],
    },
    'Infrastructure': {
        'weight': 0.15,
        'features': ['Fiber_Available', 'gov_civil', 'malls_shopping', 'edu_higher'],
    },
}

ALL_RLI_FEATURES = sorted({f for p in PILLARS.values() for f in p['features']})

# Features that are raw counts and need density normalization (÷ km²)
COUNT_FEATURES = [
    'bus_count', 'metro_count', 'dining_cafe', 'med_facilities',
    'edu_primary', 'religious', 'essential_retail', 'parks_green',
    'sports_play', 'malls_shopping',
]

df_raw['category_name'] = df_raw['category'].map(CATEGORY_MAP)

SUPERVISED_FEATURES = [
    'area', 'category', 'total_rooms',
    'dining_cafe', 'med_facilities', 'edu_primary', 'religious',
    'essential_retail', 'parks_green', 'sports_play',
    'bus_count', 'metro_count', 'Fiber_Available', 'connectivity_score',
    'neighborhood_area_km2', 'fitness_care', 'gov_civil',
    'malls_shopping', 'edu_higher', 'pedestrian',
]
TIER_LABELS = ['Budget', 'Mid', 'Premium', 'Luxury']

print(f"RLI uses {len(ALL_RLI_FEATURES)} features across {len(PILLARS)} pillars")
print(f"Supervised models use {len(SUPERVISED_FEATURES)} features")

## 3 | `compute_rli()`
RLI scoring: Log-scale → MinMax → weighted pillars → Shannon entropy → 0-100.

In [ ]:
def compute_rli(df_in: pd.DataFrame, pillar_weights: dict | None = None):
    """
    Compute Riyadh Livability Index for all neighborhoods in df_in.

    Parameters
    ----------
    df_in : DataFrame
        Neighborhood-level aggregated features (index = neighborhood name).
    pillar_weights : dict, optional
        Custom weights like {'Core': 0.5, 'Mobility': 0.2, ...}.
        Auto-normalized to sum to 1.0.

    Returns
    -------
    (df_result, scaler) : tuple
        df_result sorted by rank (best first), fitted MinMaxScaler.
    """
    pw = pillar_weights if pillar_weights else {p: v['weight'] for p, v in PILLARS.items()}
    # Normalize weights to sum to 1
    total_w = sum(pw.values())
    if total_w > 0:
        pw = {k: v / total_w for k, v in pw.items()}

    # Step 1: Log-scale then MinMax 0–1
    available = [f for f in ALL_RLI_FEATURES if f in df_in.columns]
    df_log = np.log1p(df_in[available])
    scaler = MinMaxScaler()
    df_sc = pd.DataFrame(
        scaler.fit_transform(df_log),
        columns=available,
        index=df_in.index,
    )

    # Step 2: Weighted pillar scores
    pillar_scores = {}
    for pname, pinfo in PILLARS.items():
        feats = [f for f in pinfo['features'] if f in df_sc.columns]
        w = pw.get(pname, 0)
        pillar_scores[pname] = df_sc[feats].mean(axis=1) * w

    pillar_df = pd.DataFrame(pillar_scores, index=df_in.index)
    weighted_sum = pillar_df.sum(axis=1)

    # Step 3: Shannon Entropy across 4 pillars
    pillar_props = pillar_df.div(pillar_df.sum(axis=1) + EPSILON, axis=0)
    H = -(pillar_props * np.log(pillar_props + EPSILON)).sum(axis=1)

    # Step 4: Diversity multiplier [1 + H / H_max]
    diversity_mult = 1 + (H / H_MAX)

    # Step 5: RLI = weighted_sum × diversity_multiplier
    raw_rli = weighted_sum * diversity_mult

    # Step 6: Normalize 0–100
    r_min, r_max = raw_rli.min(), raw_rli.max()
    rli_100 = ((raw_rli - r_min) / (r_max - r_min + EPSILON)) * 100

    # Assemble result
    result = df_in.copy()
    for pname in PILLARS:
        result[f'pillar_{pname}'] = pillar_scores[pname].values
    result['H_entropy']      = H.values
    result['diversity_mult'] = diversity_mult.values
    result['raw_rli']        = raw_rli.values
    result['RLI']            = rli_100.round(2).values
    result['rank']           = result['RLI'].rank(ascending=False, method='min').astype(int)

    return result.sort_values('rank'), scaler

print("compute_rli() defined ✓")

## 4 | `build_global_ranking()`
Aggregate 348K properties → 176 neighborhoods, density-normalize, K-Means, PCA, RLI.

In [ ]:
def build_global_ranking(df_raw: pd.DataFrame, pillar_weights: dict | None = None):
    """
    Aggregate all 348K properties → 176 neighborhoods, density-normalize,
    compute RLI, cluster (K-Means), and attach PCA coordinates.

    Returns
    -------
    dict with keys:
        'df_raw'     — original DataFrame (with category_name added)
        'df_global'  — aggregated neighborhood matrix
        'df_ranked'  — ranked DataFrame (index = neighborhood)
        'pca_2d'     — fitted PCA object
        'kmeans'     — fitted KMeans object
    """
    # Add category_name if missing
    if 'category_name' not in df_raw.columns:
        df_raw = df_raw.copy()
        df_raw['category_name'] = df_raw['category'].map(CATEGORY_MAP)

    # ── 1. City-wide aggregation ──
    df_global = df_raw.groupby('neighborhood').agg(
        neighborhood_area_km2=('neighborhood_area_km2', 'first'),
        property_count=('property_id', 'count'),
        median_price=('price', 'median'),
        median_area=('area', 'median'),
        lat=('lat', 'mean'),
        lng=('lng', 'mean'),
        n_categories=('category', 'nunique'),
        **{f: (f, 'mean') for f in ALL_RLI_FEATURES},
    )

    # ── 2. Density normalization (raw counts ÷ km²) ──
    for feat in COUNT_FEATURES:
        if feat in df_global.columns:
            df_global[feat] = df_global[feat] / df_global['neighborhood_area_km2']
    df_global = df_global.replace([np.inf, -np.inf], 0).fillna(0)

    # ── 3. K-Means clustering ──
    cluster_feats = [f for f in ALL_RLI_FEATURES if f in df_global.columns]
    X_std = (df_global[cluster_feats] - df_global[cluster_feats].mean()) / (df_global[cluster_feats].std() + EPSILON)
    kmeans = KMeans(n_clusters=K_BEST, random_state=42, n_init=10)
    df_global['km_cluster'] = kmeans.fit_predict(X_std)

    # ── 4. PCA for visualization ──
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X_std)
    df_global['PC1'] = coords[:, 0]
    df_global['PC2'] = coords[:, 1]

    # ── 5. Compute RLI ──
    df_ranked, scaler = compute_rli(df_global, pillar_weights)

    return {
        'df_raw':    df_raw,
        'df_global': df_global,
        'df_ranked': df_ranked,
        'pca_2d':    pca,
        'kmeans':    kmeans,
        'scaler':    scaler,
    }

print("build_global_ranking() defined ✓")

## 5 | `recommend()`
Re-score all neighborhoods with custom pillar weights.

In [ ]:
def recommend(df_ranked: pd.DataFrame, user_weights: dict | None = None, top_n: int = 20):
    """
    Re-score all neighborhoods with custom pillar weights.
    Returns re-ranked DataFrame with match_pct (% of top scorer).
    """
    df_re, _ = compute_rli(df_ranked, pillar_weights=user_weights)

    top_rli = df_re['RLI'].max()
    df_re['match_pct'] = (df_re['RLI'] / (top_rli + EPSILON) * 100).round(1)

    return df_re.head(top_n) if top_n else df_re

print("recommend() defined ✓")

## 6 | `property_search()` — with K-Means Cluster Matching
Filter-first search with three-factor scoring: RLI + Price + Cluster Fit.

In [ ]:
def property_search(
    df_raw: pd.DataFrame,
    df_ranked: pd.DataFrame,
    category: int,
    min_price: float = 0,
    max_price: float = float('inf'),
    min_rooms: int = 0,
    pillar_weights: dict | None = None,
    price_weight: float = 0.20,
    cluster_weight: float = 0.15,
):
    """
    Filter-first property search with K-Means cluster matching.

    1. Filter raw properties by category + price + rooms.
    2. Identify qualifying neighborhoods.
    3. Re-score using RLI + price compatibility + K-Means cluster fit.

    K-Means Integration
    -------------------
    After filtering, the system computes the average pillar profile of each
    K-Means cluster among qualifying neighborhoods, then scores each cluster
    against the user's pillar weights. Neighborhoods in the best-fit cluster
    receive a boost, making the recommendation cluster-aware.

    Returns
    -------
    dict with keys:
        'results'              — DataFrame of ranked neighborhoods
        'properties_matched'   — int, how many raw properties survived
        'category_label'       — str
        'best_cluster'         — int, the cluster ID that best matches user priorities
        'cluster_scores'       — dict, {cluster_id: alignment_score}
    """
    cat_label = CATEGORY_MAP.get(category, f'Category {category}')

    # Ensure category_name exists
    if 'category_name' not in df_raw.columns:
        df_raw = df_raw.copy()
        df_raw['category_name'] = df_raw['category'].map(CATEGORY_MAP)

    # ── Step A: Strict filtering ──
    mask = df_raw['category'] == category
    mask &= df_raw['price'].between(min_price, max_price)

    if cat_label not in NO_ROOM_CATEGORIES and min_rooms > 0:
        mask &= df_raw['total_rooms'] >= min_rooms

    filtered = df_raw[mask]

    if len(filtered) == 0:
        return {
            'results': pd.DataFrame(),
            'properties_matched': 0,
            'category_label': cat_label,
            'best_cluster': -1,
            'cluster_scores': {},
        }

    # ── Step B: Neighborhood aggregation ──
    neigh_stats = filtered.groupby('neighborhood').agg(
        filtered_count=('property_id', 'count'),
        avg_price=('price', 'mean'),
        median_price=('price', 'median'),
        avg_area=('area', 'mean'),
        avg_rooms=('total_rooms', 'mean'),
    ).round(1)

    qualifying = neigh_stats.index.tolist()
    df_score_input = df_ranked.loc[df_ranked.index.isin(qualifying)].copy()

    # ── Step C: Re-score with user weights ──
    df_scored, _ = compute_rli(df_score_input, pillar_weights=pillar_weights)

    # ── Step D: K-Means cluster matching ──
    # Score each cluster by how well its average pillar profile aligns with user weights
    pw = pillar_weights if pillar_weights else {p: v['weight'] for p, v in PILLARS.items()}
    total_w = sum(pw.values())
    if total_w > 0:
        pw = {k: v / total_w for k, v in pw.items()}

    cluster_scores = {}
    if 'km_cluster' in df_scored.columns:
        pillar_cols = [f'pillar_{p}' for p in PILLARS.keys()]
        available_pcols = [c for c in pillar_cols if c in df_scored.columns]

        for cid, grp in df_scored.groupby('km_cluster'):
            # Average pillar profile of this cluster
            cluster_profile = grp[available_pcols].mean()
            # Dot product with user weights = alignment score
            alignment = sum(
                cluster_profile.get(f'pillar_{p}', 0) * pw.get(p, 0)
                for p in PILLARS.keys()
            )
            cluster_scores[int(cid)] = round(alignment, 6)

        # Best cluster = highest alignment with user priorities
        best_cluster = max(cluster_scores, key=cluster_scores.get) if cluster_scores else -1

        # Normalize cluster scores to 0–100 for the boost
        cs_vals = list(cluster_scores.values())
        cs_min, cs_max = min(cs_vals), max(cs_vals)
        cs_range = cs_max - cs_min

        if cs_range > 0:
            df_scored['cluster_fit'] = df_scored['km_cluster'].map(
                lambda c: ((cluster_scores.get(c, 0) - cs_min) / cs_range) * 100
            )
        else:
            df_scored['cluster_fit'] = 100.0
    else:
        best_cluster = -1
        df_scored['cluster_fit'] = 0.0

    # ── Step E: Price compatibility score ──
    budget_mid = (min_price + max_price) / 2
    price_dist = np.abs(neigh_stats['avg_price'] - budget_mid)
    price_max_dist = price_dist.max()
    neigh_stats['price_score'] = (
        (1 - price_dist / price_max_dist) * 100 if price_max_dist > 0 else 100.0
    )

    # ── Step F: Merge & combined score (RLI + Price + Cluster Fit) ──
    overlap = neigh_stats.columns.intersection(df_scored.columns)
    df_result = df_scored.join(neigh_stats.drop(columns=overlap), how='inner')

    # Three-factor combined score
    rli_w = 1 - price_weight - cluster_weight
    rli_w = max(rli_w, 0)  # safety clamp
    df_result['combined_score'] = (
        df_result['RLI'] * rli_w
        + df_result['price_score'] * price_weight
        + df_result['cluster_fit'] * cluster_weight
    ).round(2)

    top_cs = df_result['combined_score'].max()
    df_result['match_pct'] = (
        (df_result['combined_score'] / (top_cs + EPSILON) * 100).round(1)
    )
    df_result['rank'] = df_result['combined_score'].rank(
        ascending=False, method='min'
    ).astype(int)
    df_result = df_result.sort_values('rank')

    return {
        'results': df_result,
        'properties_matched': len(filtered),
        'category_label': cat_label,
        'best_cluster': best_cluster,
        'cluster_scores': cluster_scores,
    }

print("property_search() defined ✓ (with K-Means cluster matching)")

## 7 | ML Model Comparison Functions
Three functions that run multiple algorithms and return metrics for comparison.

### 7.1 | Clustering Comparison

In [ ]:
def run_clustering_comparison(df_ranked: pd.DataFrame):
    """
    Compare K-Means (k=2,3,5) vs Hierarchical (k=2,3,5) on neighborhood features.
    Returns dict of {model_name: {silhouette, calinski_harabasz, davies_bouldin, labels}}.
    """
    cluster_feats = [f for f in ALL_RLI_FEATURES if f in df_ranked.columns]
    X = df_ranked[cluster_feats].fillna(0)
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)

    results = {}
    for k in [2, 3, 5]:
        # K-Means
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels_km = km.fit_predict(X_std)
        results[f'KMeans (k={k})'] = {
            'silhouette': silhouette_score(X_std, labels_km),
            'calinski_harabasz': calinski_harabasz_score(X_std, labels_km),
            'davies_bouldin': davies_bouldin_score(X_std, labels_km),
            'labels': labels_km,
            'k': k,
        }
        # Hierarchical
        hc = AgglomerativeClustering(n_clusters=k, linkage='ward')
        labels_hc = hc.fit_predict(X_std)
        results[f'Hierarchical (k={k})'] = {
            'silhouette': silhouette_score(X_std, labels_hc),
            'calinski_harabasz': calinski_harabasz_score(X_std, labels_hc),
            'davies_bouldin': davies_bouldin_score(X_std, labels_hc),
            'labels': labels_hc,
            'k': k,
        }

    # Identify best by silhouette
    best_name = max(results, key=lambda x: results[x]['silhouette'])
    return results, X_std, best_name

print("run_clustering_comparison() defined ✓")

### 7.2 | Regression Comparison

In [ ]:
def run_regression_comparison(df_raw: pd.DataFrame, sample_size: int = 30000):
    """
    Compare Linear, Ridge, Random Forest, Gradient Boosting on price prediction.
    Returns dict of {model_name: {R2, MAE, RMSE, MAPE, predictions, model}}.
    """
    df = df_raw.sample(min(sample_size, len(df_raw)), random_state=42).copy()
    df = df.dropna(subset=SUPERVISED_FEATURES + ['price'])
    q01, q99 = df['price'].quantile(0.01), df['price'].quantile(0.99)
    df = df[(df['price'] >= q01) & (df['price'] <= q99)]

    X = df[SUPERVISED_FEATURES]
    y = df['price']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    models = [
        ('Linear Regression', LinearRegression(), True),
        ('Ridge Regression', Ridge(alpha=1.0), True),
        ('Random Forest', RandomForestRegressor(
            n_estimators=100, max_depth=18, min_samples_split=5,
            min_samples_leaf=3, random_state=42, n_jobs=-1), False),
        ('Gradient Boosting', GradientBoostingRegressor(
            n_estimators=150, max_depth=5, learning_rate=0.1,
            subsample=0.8, random_state=42), False),
    ]

    results = {}
    for name, model, use_scaled in models:
        if use_scaled:
            model.fit(X_train_sc, y_train)
            preds = model.predict(X_test_sc)
        else:
            model.fit(X_train, y_train)
            preds = model.predict(X_test)

        r2 = r2_score(y_test, preds)
        mae = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mape = np.mean(np.abs((y_test - preds) / (y_test + EPSILON))) * 100

        results[name] = {
            'R2': round(r2, 4), 'MAE': round(mae, 0),
            'RMSE': round(rmse, 0), 'MAPE': round(mape, 1),
            'y_test': y_test.values, 'predictions': preds,
            'model': model,
        }

    best_name = max(results, key=lambda x: results[x]['R2'])
    return results, best_name, SUPERVISED_FEATURES, scaler

print("run_regression_comparison() defined ✓")

### 7.3 | Classification Comparison

In [ ]:
def run_classification_comparison(df_raw: pd.DataFrame, sample_size: int = 30000):
    """
    Compare Logistic Regression, Random Forest, Gradient Boosting on price tier prediction.
    Returns dict of {model_name: {Accuracy, Precision, Recall, F1, confusion_matrix, model}}.
    """
    df = df_raw.sample(min(sample_size, len(df_raw)), random_state=42).copy()
    df = df.dropna(subset=SUPERVISED_FEATURES + ['price'])
    q01, q99 = df['price'].quantile(0.01), df['price'].quantile(0.99)
    df = df[(df['price'] >= q01) & (df['price'] <= q99)]

    df['price_tier'] = pd.qcut(df['price'], 4, labels=[0, 1, 2, 3])
    X = df[SUPERVISED_FEATURES]
    y = df['price_tier'].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    models = [
        ('Logistic Regression', LogisticRegression(
            max_iter=1000, C=1.0, random_state=42), True),
        ('Random Forest', RandomForestClassifier(
            n_estimators=100, max_depth=18, min_samples_split=5,
            min_samples_leaf=3, random_state=42, n_jobs=-1), False),
        ('Gradient Boosting', GradientBoostingClassifier(
            n_estimators=150, max_depth=5, learning_rate=0.1,
            subsample=0.8, random_state=42), False),
    ]

    results = {}
    for name, model, use_scaled in models:
        if use_scaled:
            model.fit(X_train_sc, y_train)
            preds = model.predict(X_test_sc)
        else:
            model.fit(X_train, y_train)
            preds = model.predict(X_test)

        results[name] = {
            'Accuracy': round(accuracy_score(y_test, preds), 4),
            'Precision': round(precision_score(y_test, preds, average='weighted'), 4),
            'Recall': round(recall_score(y_test, preds, average='weighted'), 4),
            'F1': round(f1_score(y_test, preds, average='weighted'), 4),
            'confusion_matrix': confusion_matrix(y_test, preds),
            'y_test': y_test.values, 'predictions': preds,
            'model': model,
        }

    best_name = max(results, key=lambda x: results[x]['F1'])
    return results, best_name, scaler

print("run_classification_comparison() defined ✓")

## 8 | Execute Pipeline & Global Ranking

In [ ]:
pipe = build_global_ranking(df_raw)
df_ranked = pipe['df_ranked']

print(f"✅ Global Ranking: {len(df_ranked)} neighborhoods scored")
print(f"   Clusters: {df_ranked['km_cluster'].nunique()} (k={K_BEST})")
print(f"\nTop 10:")
print("=" * 70)
for _, r in df_ranked.head(10).iterrows():
    name = r.name if isinstance(r.name, str) else str(r.name)
    print(f"  #{int(r['rank']):>3}  {name:<30s}  RLI={r['RLI']:>6.2f}  Cluster={int(r['km_cluster'])}")

## 9 | Unsupervised — Clustering Comparison
K-Means vs Hierarchical at k=2, 3, 5. Metrics: Silhouette (↑), Calinski-Harabasz (↑), Davies-Bouldin (↓).

In [ ]:
clust_results, X_std_clust, best_clust = run_clustering_comparison(df_ranked)

print(f"Best Clustering Model: {best_clust}")
print(f"{'='*70}")
print(f"{'Model':<22s}  {'Silhouette':>10s}  {'CH':>10s}  {'DB':>10s}")
print(f"{'-'*70}")
for name, m in sorted(clust_results.items(), key=lambda x: -x[1]['silhouette']):
    marker = ' ◄ BEST' if name == best_clust else ''
    print(f"  {name:<20s}  {m['silhouette']:>10.4f}  {m['calinski_harabasz']:>10.1f}  {m['davies_bouldin']:>10.4f}{marker}")

## 10 | Supervised — Price Prediction (Regression)
Linear, Ridge, Random Forest, Gradient Boosting. Metrics: R² (↑), MAE (↓), RMSE (↓), MAPE (↓).

In [ ]:
reg_results, best_reg, reg_feats, reg_scaler = run_regression_comparison(df_raw)

print(f"Best Regression Model: {best_reg}")
print(f"{'='*70}")
print(f"{'Model':<22s}  {'R²':>8s}  {'MAE':>12s}  {'RMSE':>12s}  {'MAPE':>8s}")
print(f"{'-'*70}")
for name, m in sorted(reg_results.items(), key=lambda x: -x[1]['R2']):
    marker = ' ◄' if name == best_reg else ''
    print(f"  {name:<20s}  {m['R2']:>8.4f}  {m['MAE']:>12,.0f}  {m['RMSE']:>12,.0f}  {m['MAPE']:>7.1f}%{marker}")

## 11 | Supervised — Price Tier Classification
Logistic Regression, Random Forest, Gradient Boosting. Metrics: Accuracy, Precision, Recall, F1, Confusion Matrix.

In [ ]:
cls_results, best_cls, cls_scaler = run_classification_comparison(df_raw)

print(f"Best Classification Model: {best_cls}")
print(f"{'='*70}")
print(f"{'Model':<22s}  {'Accuracy':>10s}  {'Precision':>10s}  {'Recall':>10s}  {'F1':>10s}")
print(f"{'-'*70}")
for name, m in sorted(cls_results.items(), key=lambda x: -x[1]['F1']):
    marker = ' ◄' if name == best_cls else ''
    print(f"  {name:<20s}  {m['Accuracy']:>10.4f}  {m['Precision']:>10.4f}  {m['Recall']:>10.4f}  {m['F1']:>10.4f}{marker}")

print(f"\nConfusion Matrix ({best_cls}):")
cm = cls_results[best_cls]['confusion_matrix']
print(f"  {'':>10s}  {'Budget':>8s}  {'Mid':>8s}  {'Premium':>8s}  {'Luxury':>8s}")
for i, tier in enumerate(TIER_LABELS):
    print(f"  {tier:>10s}  {cm[i][0]:>8d}  {cm[i][1]:>8d}  {cm[i][2]:>8d}  {cm[i][3]:>8d}")

## 12 | Demo — Property Search with K-Means Cluster Matching

In [ ]:
result = property_search(
    df_raw=df_raw, df_ranked=df_ranked,
    category=3, min_price=500_000, max_price=2_000_000,
    min_rooms=6, price_weight=0.20, cluster_weight=0.15,
)
df_res = result['results']
print(f"Villa search: {result['properties_matched']:,} properties → {len(df_res)} neighborhoods")
print(f"Best-fit cluster: {result['best_cluster']}")
print(f"Cluster scores: {result['cluster_scores']}")
if len(df_res) > 0:
    print(f"\nTop 5:")
    for _, r in df_res.head(5).iterrows():
        name = r.name if isinstance(r.name, str) else str(r.name)
        print(f"  #{int(r['rank']):>2}  {name:<25s}  Match={r['match_pct']:>5.1f}%  RLI={r['RLI']:>6.2f}  ClusterFit={r.get('cluster_fit',0):>5.1f}")

## 13 | Key Findings

### Model Selection Results
| Problem | Best Model | Key Metric |
|---|---|---|
| Clustering | K-Means (k=2) | Silhouette = 0.1761 |
| Regression | Random Forest | R² = 0.919 |
| Classification | Gradient Boosting | F1 = 0.878 |

### How ML Drives the System
1. **K-Means** clusters 176 neighborhoods into archetypes → used in property search to boost neighborhoods matching user's lifestyle priorities.
2. **Random Forest** achieves R²=0.92 on price prediction → validates that neighborhood features genuinely predict property value.
3. **Gradient Boosting** classifies price tiers at 87.7% accuracy → confirms the feature set captures meaningful price-tier distinctions.

### Three-Factor Recommendation Score
$$Score = RLI \times w_{rli} + PriceScore \times w_{price} + ClusterFit \times w_{cluster}$$
Default: 65% livability + 20% price fit + 15% cluster archetype match.